In [1]:
import math
import time
from array import array
from itertools import zip_longest
from pathlib import Path
import torch
import torch.nn as nn
import sentencepiece as spm
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Sampler

#### 加载分词器

In [2]:
# 支持从项目根目录或 Notebook 所在目录启动。
project_dir = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/model/tokenizers/zh_model.model").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("请从项目目录内启动 Notebook")
model_dir = project_dir / "src/model/tokenizers"
zh_sp = spm.SentencePieceProcessor(model_file=str(model_dir / "zh_model.model"))
en_sp = spm.SentencePieceProcessor(model_file=str(model_dir / "en_model.model"))
src_vocab_size = zh_sp.get_piece_size()
tgt_vocab_size = en_sp.get_piece_size()

#### 实现位置编码

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("当前实现要求 d_model 为偶数")
        self.dropout = nn.Dropout(dropout)
        # 创建位置编码矩阵
        pe = torch.zeros(max_len, d_model)
        # 生成位置索引
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        # 计算不同维度对应的频率
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        # 偶数维使用 sin，奇数维使用 cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 增加批次维度：(1, max_len, d_model)
        pe = pe.unsqueeze(0)
        # 保存为非训练参数的 buffer
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x 的形状：(B, L, D)
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            raise ValueError("序列长度超过 max_len")
        # 添加位置信息，再进行 Dropout
        x = x + self.pe[:, :seq_len, :]

        return self.dropout(x)

#### 拼接我们的模型

In [4]:
class TranslationTransformer(nn.Module):
    def __init__(self,src_vocab_size,tgt_vocab_size,d_model=256,nhead=8,num_encoder_layers=3,
        num_decoder_layers=3,dim_feedforward=512,dropout=0.1,max_len=512,src_pad_id=0,tgt_pad_id=0):
        super().__init__()
        self.d_model = d_model
        # 中文和英文的 Token Embedding
        self.src_embedding = nn.Embedding(src_vocab_size,d_model,padding_idx=src_pad_id)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size,d_model,padding_idx=tgt_pad_id)

        # 正余弦位置编码
        self.src_pos_encoding = PositionalEncoding(d_model=d_model,dropout=dropout,max_len=max_len)
        self.tgt_pos_encoding = PositionalEncoding(d_model=d_model,dropout=dropout,max_len=max_len)

        # 调用 PyTorch 官方 Transformer
        self.transformer = nn.Transformer(d_model=d_model,nhead=nhead,num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,dim_feedforward=dim_feedforward,dropout=dropout,batch_first=True)
        # 将特征向量映射到英文词表
        self.output_projection = nn.Linear(d_model,tgt_vocab_size)

    def forward(self,src,decoder_input,src_padding_mask=None,tgt_padding_mask=None,causal_mask=None):
        # src: (B, S)
        # decoder_input: (B, T)
        # Token ID -> Embedding
        src_emb = self.src_embedding(src)
        tgt_emb = self.tgt_embedding(decoder_input)

        # 按原始 Transformer 的缩放方式处理 Embedding
        scale = math.sqrt(self.d_model)
        src_emb = src_emb * scale
        tgt_emb = tgt_emb * scale

        # 添加位置编码
        src_emb = self.src_pos_encoding(src_emb)
        tgt_emb = self.tgt_pos_encoding(tgt_emb)

        # 经过 Encoder 和 Decoder
        output = self.transformer(src=src_emb,tgt=tgt_emb,tgt_mask=causal_mask,src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,memory_key_padding_mask=src_padding_mask)
        # 映射到英文词表的每个 Token
        logits = self.output_projection(output)

        return logits

In [5]:
# 创建模型
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TranslationTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=256,
    nhead=4,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=512,
    dropout=0.1,
    max_len=1024,
    src_pad_id=zh_sp.pad_id(),
    tgt_pad_id=en_sp.pad_id()
).to(device)


In [6]:
print("项目根目录：", project_dir)
data_dir = project_dir / "data" / "sample_for_train_val_test"

class TranslationDataset(Dataset):
    def __init__(self, zh_path, en_path, zh_sp, en_sp, max_len=128):
        if max_len < 2:
            raise ValueError("max_len 至少为 2，以容纳 BOS/EOS")
        self.max_len = max_len
        src_buffer, tgt_buffer = array("q"), array("q")
        self.src_offsets, self.tgt_offsets = array("q", [0]), array("q", [0])
        self.lengths = array("q")
        src_bos, src_eos = zh_sp.bos_id(), zh_sp.eos_id()
        tgt_bos, tgt_eos = en_sp.bos_id(), en_sp.eos_id()

        # 流式读取，避免同时保留全部原文和大量 Python 整数对象。
        with open(zh_path, encoding="utf-8") as zh_file, open(en_path, encoding="utf-8") as en_file:
            for zh, en in zip_longest(zh_file, en_file):
                if zh is None or en is None:
                    raise ValueError("中英文行数不一致！")
                src_ids = [src_bos] + zh_sp.encode(zh.strip(), out_type=int)[:max_len - 2] + [src_eos]
                tgt_ids = [tgt_bos] + en_sp.encode(en.strip(), out_type=int)[:max_len - 2] + [tgt_eos]
                src_buffer.extend(src_ids)
                tgt_buffer.extend(tgt_ids)
                self.src_offsets.append(len(src_buffer))
                self.tgt_offsets.append(len(tgt_buffer))
                self.lengths.append(max(len(src_ids), len(tgt_ids)))

        # frombuffer 共享连续存储，并持有底层 buffer；之后不再扩容或修改。
        self.src_tokens = torch.frombuffer(src_buffer, dtype=torch.long) if src_buffer else torch.empty(0, dtype=torch.long)
        self.tgt_tokens = torch.frombuffer(tgt_buffer, dtype=torch.long) if tgt_buffer else torch.empty(0, dtype=torch.long)

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, index):
        return (
            self.src_tokens[self.src_offsets[index]:self.src_offsets[index + 1]],
            self.tgt_tokens[self.tgt_offsets[index]:self.tgt_offsets[index + 1]],
        )


class LengthBucketBatchSampler(Sampler):
    """训练时随机分池、池内按长度组批，再打乱批次；每轮覆盖每条样本一次。"""
    def __init__(self, lengths, batch_size=32, shuffle=True, pool_batches=50, seed=42):
        if batch_size < 1 or pool_batches < 1:
            raise ValueError("batch_size 和 pool_batches 必须为正数")
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.pool_size = batch_size * pool_batches
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        if self.shuffle:
            indices = torch.randperm(len(self.lengths), generator=generator).tolist()
        else:
            indices = sorted(range(len(self.lengths)), key=self.lengths.__getitem__)
        for start in range(0, len(indices), self.pool_size):
            pool = sorted(indices[start:start + self.pool_size], key=self.lengths.__getitem__)
            batches = [pool[i:i + self.batch_size] for i in range(0, len(pool), self.batch_size)]
            order = torch.randperm(len(batches), generator=generator).tolist() if self.shuffle else range(len(batches))
            for index in order:
                yield batches[index]

项目根目录： d:\programs\Transformer-translate


In [7]:
train_dataset = TranslationDataset(
    data_dir / "train_zh.txt",
    data_dir / "train_en.txt",
    zh_sp, en_sp
)

val_dataset = TranslationDataset(
    data_dir / "val_zh.txt",
    data_dir / "val_en.txt",
    zh_sp, en_sp
)

print("训练样本数：", len(train_dataset))
print("验证样本数：", len(val_dataset))

训练样本数： 450000
验证样本数： 25000


In [8]:
def collate_fn(batch):
    src_sequences, tgt_sequences = zip(*batch)
    # 样本已经是 Tensor 视图；只分配本批次的补齐结果。
    src = pad_sequence(src_sequences, batch_first=True, padding_value=zh_sp.pad_id())
    tgt_full = pad_sequence(tgt_sequences, batch_first=True, padding_value=en_sp.pad_id())
    decoder_input = tgt_full[:, :-1]
    labels = tgt_full[:, 1:].clone()
    labels.masked_fill_(labels == en_sp.pad_id(), -100)
    # Padding mask 在目标设备上生成；causal mask 在每轮中复用。
    return {"src": src, "decoder_input": decoder_input, "labels": labels}

In [9]:
batch_size = 32
train_batch_sampler = LengthBucketBatchSampler(train_dataset.lengths, batch_size=batch_size)
val_batch_sampler = LengthBucketBatchSampler(val_dataset.lengths, batch_size=batch_size, shuffle=False)

# Windows Notebook 保持单进程加载，避免复制数据集及增加 CPU 并发。
train_loader = DataLoader(train_dataset, batch_sampler=train_batch_sampler, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_sampler=val_batch_sampler, collate_fn=collate_fn, num_workers=0)
print("训练 Batch 数：", len(train_loader))
print("验证 Batch 数：", len(val_loader))

训练 Batch 数： 14063
验证 Batch 数： 782


In [10]:
batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(
        f"{name:20s}",
        "shape =", tuple(tensor.shape),
        "dtype =", tensor.dtype
    )

src                  shape = (32, 10) dtype = torch.int64
decoder_input        shape = (32, 9) dtype = torch.int64
labels               shape = (32, 9) dtype = torch.int64


In [11]:
def run_epoch(model, loader, device, src_pad_id, tgt_pad_id, optimizer=None, log_interval=50):
    """训练/验证共用循环；保持 FP32 和原来的逐有效 Token 平均损失。"""
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss(ignore_index=-100, reduction="sum")
    loss_total = torch.zeros((), dtype=torch.float64, device=device)
    token_total = torch.zeros((), dtype=torch.long, device=device)
    max_target_len = loader.dataset.max_len - 1
    causal_mask = torch.ones(max_target_len, max_target_len, dtype=torch.bool, device=device).triu_(1)

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    start_time = time.perf_counter()

    with torch.set_grad_enabled(training):
        for step, batch in enumerate(loader, start=1):
            # 前向计算之前释放上一个批次的梯度。
            if training:
                optimizer.zero_grad(set_to_none=True)
            src = batch["src"].to(device)
            decoder_input = batch["decoder_input"].to(device)
            labels = batch["labels"].to(device)
            target_len = decoder_input.size(1)
            logits = model(
                src=src,
                decoder_input=decoder_input,
                src_padding_mask=src.eq(src_pad_id),
                tgt_padding_mask=decoder_input.eq(tgt_pad_id),
                causal_mask=causal_mask[:target_len, :target_len],
            )
            loss_sum = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
            token_count = labels.ne(-100).sum()
            loss = loss_sum / token_count.clamp_min(1)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # detach 后累计，避免保留计算图；仅输出日志和轮末读取 GPU 标量。
            loss_total.add_(loss_sum.detach())
            token_total.add_(token_count)
            if training and log_interval and (step % log_interval == 0 or step == len(loader)):
                print(f"Step {step}/{len(loader)} | 当前 Batch Loss: {loss.item():.4f}")
            # 防止上一批次的 logits 与下一次前向计算同时占用显存。
            del logits, loss_sum, loss, token_count, src, decoder_input, labels

    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = loss_total.item()
    total_tokens = token_total.item()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start_time
    return {
        "loss": total_loss / max(total_tokens, 1),
        "tokens": total_tokens,
        "seconds": elapsed,
        "tokens_per_second": total_tokens / max(elapsed, 1e-9),
        "peak_memory_mb": torch.cuda.max_memory_allocated(device) / 1024 ** 2 if device.type == "cuda" else None,
    }

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
checkpoint_dir = project_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_val_loss = float("inf")
num_epochs = 10

# 不再额外执行一次演示训练，避免正式训练前更新模型并重建优化器。
for epoch in range(num_epochs):
    start_time = time.perf_counter()
    train_batch_sampler.set_epoch(epoch)
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_metrics = run_epoch(model, train_loader, device, zh_sp.pad_id(), en_sp.pad_id(), optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, device, zh_sp.pad_id(), en_sp.pad_id())
    train_loss, val_loss = train_metrics["loss"], val_metrics["loss"]

    print(f"\nEpoch {epoch + 1} 结果")
    print(f"训练损失：{train_loss:.4f} | 验证损失：{val_loss:.4f}")
    print(f"训练有效 Token/s：{train_metrics['tokens_per_second']:.0f}")
    if train_metrics["peak_memory_mb"] is not None:
        print(f"训练峰值显存（已分配）：{train_metrics['peak_memory_mb']:.1f} MiB")
    print(f"耗时：{time.perf_counter() - start_time:.1f} 秒")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "src_vocab_size": src_vocab_size,
                "tgt_vocab_size": tgt_vocab_size,
            },
            checkpoint_dir / "best_transformer.pt",
        )
        print("已保存当前最佳模型。")

Epoch 1/1
Step 50/14063 | 当前 Batch Loss: 2.6339
Step 100/14063 | 当前 Batch Loss: 2.9946
Step 150/14063 | 当前 Batch Loss: 2.4793
Step 200/14063 | 当前 Batch Loss: 2.8079
Step 250/14063 | 当前 Batch Loss: 2.5556
Step 300/14063 | 当前 Batch Loss: 2.9939
Step 350/14063 | 当前 Batch Loss: 2.4665
Step 400/14063 | 当前 Batch Loss: 2.3622
Step 450/14063 | 当前 Batch Loss: 3.1001
Step 500/14063 | 当前 Batch Loss: 1.5480
Step 550/14063 | 当前 Batch Loss: 1.0897
Step 600/14063 | 当前 Batch Loss: 2.6047
Step 650/14063 | 当前 Batch Loss: 0.2432
Step 700/14063 | 当前 Batch Loss: 2.6023
Step 750/14063 | 当前 Batch Loss: 2.4551
Step 800/14063 | 当前 Batch Loss: 2.6388
Step 850/14063 | 当前 Batch Loss: 1.7988
Step 900/14063 | 当前 Batch Loss: 2.7396
Step 950/14063 | 当前 Batch Loss: 2.7231
Step 1000/14063 | 当前 Batch Loss: 2.7467
Step 1050/14063 | 当前 Batch Loss: 2.9035
Step 1100/14063 | 当前 Batch Loss: 1.6352
Step 1150/14063 | 当前 Batch Loss: 1.1930
Step 1200/14063 | 当前 Batch Loss: 2.7863
Step 1250/14063 | 当前 Batch Loss: 0.3468
Step 1300/